# Breast Cancer Wisconsin Dataset: análisis con SVM

En este notebook vamos a:

1. Cargar y entender el dataset.
2. Explorar las variables (features).
3. Analizar cuáles parecen más informativas.
4. Seleccionar **2 features** para poder visualizar en 2D.
5. Entrenar varios modelos **SVM** con distintos kernels.
6. Comparar resultados y visualizar fronteras de decisión.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import mutual_info_classif

plt.rcParams['figure.figsize'] = (8, 5)

## 2. Carga del dataset

Este dataset viene incluido en `scikit-learn` y contiene información sobre tumores de mama.

- **569 observaciones**
- **30 features numéricas**
- **target binario**:
  - `0` = malignant
  - `1` = benign

Podemos ver una descripción más detallada en https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

In [ ]:
cancer = load_breast_cancer()

X = cancer.data
y = cancer.target

print('Shape de X:', X.shape)
print('Shape de y:', y.shape)
print('Clases:', cancer.target_names)

In [ ]:
df = pd.DataFrame(X, columns=cancer.feature_names)
df['target'] = y
df.head()

## 3. Entender las variables

Las variables son medidas calculadas a partir de imágenes de núcleos celulares.

Por ejemplo:
- `mean radius`
- `mean texture`
- `mean perimeter`
- `mean area`

También hay versiones `worst` (núcleos más grandes) y errores estándar `error`.

In [ ]:
pd.Series(cancer.feature_names, name='features')

## 4. Análisis exploratorio rápido

In [ ]:
df.describe().T[['mean', 'std', 'min', 'max']].head(10)

In [ ]:
class_counts = pd.Series(y).value_counts().sort_index()
class_labels = pd.Series(y).map({0: 'malignant', 1: 'benign'})

print('Distribución de clases:')
print(class_labels.value_counts())

## 5. ¿Qué features parecen más útiles, si tuviéramos que elegir alguna?

Vamos a usar **dos criterios sencillos** para estimar qué features pueden ser de mayor provecho:

1. **Diferencia de medias por clase**: una feature que cambia mucho entre benigno y maligno puede ser útil.
2. **Mutual information**: mide cuánta información aporta una feature respecto a la clase objetivo.

Ambas métricas nos dan una idea de la *separabilidad* de las clases.

In [ ]:
# Diferencia absoluta entre medias por clase
group_means = df.groupby('target').mean().T
group_means.columns = ['malignant_mean', 'benign_mean']
group_means['abs_mean_diff'] = (group_means['malignant_mean'] - group_means['benign_mean']).abs()

# Mutual information
mi = mutual_info_classif(cancer.data, y, random_state=42)
group_means['mutual_info'] = mi

# Ranking combinado simple
group_means['rank_abs_diff'] = group_means['abs_mean_diff'].rank(ascending=False)
group_means['rank_mi'] = group_means['mutual_info'].rank(ascending=False)
group_means['combined_rank'] = group_means['rank_abs_diff'] + group_means['rank_mi']

feature_ranking = group_means.sort_values('combined_rank')
feature_ranking.head(10)

In [ ]:
top_features = feature_ranking.head(4).index.tolist()
top_features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, feature in zip(axes, top_features):
    ax.hist(df.loc[df['target'] == 0, feature], bins=25, alpha=0.6, label='malignant')
    ax.hist(df.loc[df['target'] == 1, feature], bins=25, alpha=0.6, label='benign')
    ax.set_title(feature)
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Selección de 2 features para el plano 2D

Vamos a dibujar la frontera de decisión en 2D, con lo que entrenaremos el modelo con tan solo 2 features.

Tomaremos las **2 primeras del ranking combinado**.

In [ ]:
# pair plot of sample feature
sns.pairplot(df, hue = 'target', 
             vars = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness'] )

In [ ]:
selected_features = feature_ranking.head(2).index.tolist()
selected_features

In [ ]:
X_2d = df[selected_features].values

plt.figure(figsize=(8, 6))
plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=y,
    alpha=0.75
)
plt.xlabel(selected_features[0])
plt.ylabel(selected_features[1])
plt.title('Visualización 2D con las dos features seleccionadas')
plt.show()

## 8. Train / test split y escalado

En SVM (en general, en métodos basados en distancia), especialmente con kernels como `rbf`, el **escalado** es importante.

Por eso:
- separamos train/test
- ajustamos el `StandardScaler` **solo con train**
- transformamos train y test con ese mismo escalador

El ajuste del scaler con train se realiza para evitar fuga de datos (el modelo usaría info estadística del training set al evaluar en el test set)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_2d, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('X_train:', X_train_scaled.shape)
print('X_test:', X_test_scaled.shape)

## 9. Entrenamiento con varios kernels

Vamos a comparar:

- `linear`
- `poly`
- `rbf`
- `sigmoid`

Todos usan las mismas dos features.

In [ ]:
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
results = []
models = {}

for kernel in kernels:
    model = SVC(kernel=kernel, C=1.0, gamma='scale')
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    
    results.append({'kernel': kernel, 'accuracy': acc})
    models[kernel] = model

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False)
results_df

## 10. Evaluación más detallada

In [ ]:
best_kernel = results_df.iloc[0]['kernel']
best_model = models[best_kernel]
best_preds = best_model.predict(X_test_scaled)

print('Mejor kernel:', best_kernel)
print('Accuracy:', accuracy_score(y_test, best_preds))

# Matriz de confusión
cm = confusion_matrix(y_test, best_preds)

fig, ax = plt.subplots(figsize=(6,5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=cancer.target_names
)

disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)

plt.title(f"Matriz de confusión - SVM ({best_kernel} kernel)")
plt.xlabel("Etiqueta predicha")
plt.ylabel("Etiqueta real")

plt.show()

print("\nInforme de clasificación:\n")
print(classification_report(y_test, best_preds, target_names=cancer.target_names))

## 11. Función para dibujar la frontera de decisión

In [ ]:
def plot_decision_boundary(model, X, y, title, xlabel, ylabel):
    plt.figure(figsize=(8, 6))
    plt.scatter(X[:, 0], X[:, 1], c=y, alpha=0.8)

    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    xx, yy = np.meshgrid(
        np.linspace(xlim[0], xlim[1], 300),
        np.linspace(ylim[0], ylim[1], 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.decision_function(grid).reshape(xx.shape)

    plt.contour(
        xx, yy, Z,
        levels=[-1, 0, 1],
        alpha=0.7,
        linestyles=['--', '-', '--']
    )

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.show()

## 12. Fronteras de decisión para cada kernel

Usaremos los datos de test ya escalados para ver cómo separa cada SVM.

In [ ]:
for kernel in kernels:
    plot_decision_boundary(
        models[kernel],
        X_test_scaled,
        y_test,
        title=f'SVM con kernel = {kernel}',
        xlabel=selected_features[0] + ' (scaled)',
        ylabel=selected_features[1] + ' (scaled)'
    )